# Explore dataset structure

In [1]:
from dirs import *

from xflow.utils.io import scan_files
from xflow.utils.sql import merge_sqlite_dbs
from xflow import SqlProvider, PyTorchPipeline
from xflow.data import build_transforms_from_config

from config_utils import load_config, detect_machine
import pandas as pd
import sqlite3
from utils import *

experiment_name = "CLEAR_visualization"  
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

c:\Users\qiyuanxu\Documents\GitHub\fiber-image-reconstruction-comparison
[config_utils] Using machine profile: win-qiyuanxu


# Load dataset

Different with training pipeline, data sample meta data is given

Mainly the run time sythetic dataset need to be compiled and generate here

In [2]:
# pip install xflow-py
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.extensions.physics import pattern_gen
from tqdm import tqdm

"""
    Contract:
    1. Unify the data format for visualization like (2, 1000, 1, 256, 256) (fiber_input_output_image_type, num_samples, C, H, W)
    2. Add labels/colors or class to data samples with index based order matter, index based.
    3. Union data samples, labels and colors into three single iterable for down stream visualization (order matter)
"""

# ============================
# DMD + orthognal basis syth data augmentation
# ============================
dmd_orth_basis_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)

dmd_validation_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs["DMD_only"]["dataset_extracted_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

val_pipeline = PyTorchPipeline(
    dmd_validation_provider, 
    transforms[:-1],
).to_numpy() 

# ========== SGM simulation pattern generator ==========
canvas = pattern_gen.DynamicPatterns(*config["simulation"]["canvas_size"])
canvas.set_postprocess_fns(build_transforms_from_config(config["simulation"]["process_functions"]))
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(config["simulation"]["total_Guassian_num"])]
canvas.set_threshold(config["simulation"]["minimum_pixel_threshold"])
stream = canvas.pattern_stream(std_1=config["simulation"]["std_1"], std_2=config["simulation"]["std_2"],
                               max_intensity=config["simulation"]["max_intensity"], fade_rate=config["simulation"]["fade_rate"], 
                               distribution=config["simulation"]["distribution"]) 

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    transforms= build_transforms_from_config(config["combinator"]["transforms"]["torch"]),
)
train_dataset = CachedBasisPipeline(
    dmd_orth_basis_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=config["data"]["total_train_samples"], 
    seed=config["seed"],
    eager=True
)

samples = []
for _ in tqdm(range(1000)):
    samples.append(next(iter(train_dataset)))
samples = np.stack(samples, axis=0)         # (1000, 2, 1, 256, 256)
samples = np.transpose(samples, (1, 0, 2, 3, 4))  # (2, 1000, 1, 256, 256)

pipeline_dmd = np.concatenate([samples, np.stack(val_pipeline, axis=0)], axis=1) 
# labels_dmd = ["DMD_syth (training set)"] * len(samples[0]) + ["DMD_val (test set)"] * len(val_pipeline[0])
# color_dmd = ["#636EFA"] * len(samples[0]) + ["#AB63FA"] * len(val_pipeline[0])  
labels_dmd = ["LED on DMD"] * len(pipeline_dmd[0])
color_dmd = ["#636EFA"] * len(pipeline_dmd[0])


# Non syth data just use the master database table, since no run time requirement
# ============================
# Chromox screen set with subsample
# ============================
chromox_provider = SqlProvider(
    sources={"connection": config["paths"]["processed_chromox"] + "/db/dataset_meta.db", "sql": config["sql"]["chromox_all"]},   
    output_config={'list': "image_path"}
).subsample(n_samples=1000, seed=config["seed"])

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": config["paths"]["processed_chromox"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

pipeline_chromox = PyTorchPipeline(
    chromox_provider, 
    transforms[:-1],
).to_numpy() 
labels_chromox = [r'$e^{-}$ Chromox'] * len(pipeline_chromox[0])
color_chromox = ["#EF553B"] * len(pipeline_chromox[0])


# ============================
# Yag screen set with subsample
# ============================
yag_provider = SqlProvider(
    sources={"connection": config["paths"]["processed_yag"] + "/db/dataset_meta.db", "sql": config["sql"]["yag_all"]},   
    output_config={'list': "image_path"}
).subsample(n_samples=1000, seed=config["seed"])

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": config["paths"]["processed_yag"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

pipeline_yag = PyTorchPipeline(
    yag_provider, 
    transforms[:-1],
).to_numpy()
labels_yag = [r'$e^{-}$ Yag'] * len(pipeline_yag[0])
color_yag = ["#00CC96"] * len(pipeline_yag[0])

# ============================
# Laser scan (Chromox) set with subsample
# ============================

laser_chromox_provider = SqlProvider(
    sources={"connection": config["paths"]["processed_chromox_laser"] + "/db/dataset_meta.db", "sql": config["sql"]["chromox_laser"]},   
    output_config={'list': "image_path"}
).subsample(n_samples=1000, seed=config["seed"])

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": config["paths"]["processed_chromox_laser"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])
pipeline_chromox_laser = PyTorchPipeline(
    laser_chromox_provider, 
    transforms[:-1],
).to_numpy()
labels_chromox_laser = ["laser on Chromox"] * len(pipeline_chromox_laser[0])
color_chromox_laser = ["#FFA15A"] * len(pipeline_chromox_laser[0])
# ============================
# Laser scan (Yag) set with subsample
# ============================

laser_yag_provider = SqlProvider(
    sources={"connection": config["paths"]["processed_yag_laser"] + "/db/dataset_meta.db", "sql": config["sql"]["yag_laser"]},   
    output_config={'list': "image_path"}
).subsample(n_samples=1000, seed=config["seed"])
config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": config["paths"]["processed_yag_laser"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])
pipeline_yag_laser = PyTorchPipeline(
    laser_yag_provider, 
    transforms[:-1],
).to_numpy()
labels_yag_laser = ["laser on Yag"] * len(pipeline_yag_laser[0])
color_yag_laser = ["#B6E880"] * len(pipeline_yag_laser[0])


# ============================
# More specific data, eg. from same domain but two type, like chromox random and chromox line scan
# ============================




--------------------------------------------------------------------------------
Connected to SQLite database with 1 tables
--------------------------------------------------------------------------------


# Latent space distribution

In [9]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path
from xflow.utils.visualization import DimReducer, Embedding3DPlot

def embed_images(images, final_method="umap", final_dim=2, random_state=42):
    X = np.asarray(images, dtype=np.float32)
    X = X.reshape(X.shape[0], -1)
    X /= (X.max() + 1e-12)

    pca = DimReducer("pca", n_components=min(50, X.shape[1]), random_state=random_state)
    X50 = pca.fit_transform(X)

    method_norm = "".join(ch for ch in str(final_method).lower() if ch.isalnum())
    if method_norm == "tsne":
        final = DimReducer(
            "tsne",
            n_components=final_dim,
            random_state=random_state,
            # initialization="pca",
            # perplexity=150,           # Default is 30. Increase to 100-250 for denser clusters.
            # early_exaggeration=24.0,  # Default is 12. Pushes distinct clusters further apart.
            # n_iter=1000               # Default is 500. Gives the algorithm more time to converge.
        )
    elif method_norm == "umap": 
        final = DimReducer(
            "umap", 
            n_components=final_dim, 
            random_state=random_state, 
            n_neighbors=10,
            min_dist=0.8, 
            metric='euclidean'
        )
    else:
        raise ValueError(f"Unsupported final_method: {final_method}")

    Z = final.fit_transform(X50)
    return Z, pca, final


# labels_dmd = ["DMD_syth (training set)"] * len(samples[0]) + ["DMD_val (test set)"] * len(val_pipeline[0])
# color_dmd = ["#636EFA"] * len(samples[0]) + ["#EF553B"] * len(val_pipeline[0])  

# Merge different types of data:
for method in ["t-SNE", "UMAP"]:  # "t-SNE", "UMAP", "openTSNE"
    index = 0  # 0: fiber output speckle, 1: original image
    group_data = {
        "dmd": (pipeline_dmd[index], labels_dmd, color_dmd),
        "chromox": (pipeline_chromox[index], labels_chromox, color_chromox),
        "yag": (pipeline_yag[index], labels_yag, color_yag),
        "chromox_laser": (pipeline_chromox_laser[index], labels_chromox_laser, color_chromox_laser),
        "yag_laser": (pipeline_yag_laser[index], labels_yag_laser, color_yag_laser),
    }

    baseline_keys = ["yag", "yag_laser"]  # "chromox", "yag", "dmd", "chromox_laser", "yag_laser"

    # Flatten the datasets, labels, and colors directly
    images = np.concatenate([group_data[k][0] for k in baseline_keys], axis=0)
    labels = [label for k in baseline_keys for label in group_data[k][1]]
    color = [c for k in baseline_keys for c in group_data[k][2]]

    # Build the 3D coordinate system
    coords, pca_model, final_model = embed_images(
        images, final_method=method, final_dim=3, random_state=42
    )

    title = f"{method.upper()} projection - {'MMF output speckle' if index == 0 else 'Original images'}"
    
    # Configure plotter
    plotter = Embedding3DPlot(
        coords=coords,
        labels=labels,
        title=title,
        point_size=2,
        color=color,
        show_projections=False,
        projection_envelope=False,
        projection_alpha=0.25,
        projection_size_scale=0.7,
        projection_gap_ratio=0.06,
        projection_envelope_alpha=0.18,
    )

    # Save orbit GIF
    out_dir = Path("results/gif")
    out_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    n_frames = 120
    base_elev, base_azim = 25.0, 40.0
    azim_radius = 25.0 / 2.0
    elev_radius = 10.0 / 2.0
    t = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

    for tt in t:
        azim = base_azim + azim_radius * np.cos(tt)
        elev = base_elev + elev_radius * np.sin(tt)

        frame = plotter.get_matplotlib_frame(
            elev=float(elev),
            azim=float(azim),
            dpi=120,
        )
        frames.append(frame)

    imageio.mimsave(out_dir / f"embedding_orbit_loop_{method}.gif", frames, fps=50, loop=0)
    plotter.close() # Clean up memory

c:\Users\qiyuanxu\AppData\Local\miniconda3\envs\torch\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


# Fiber coupling rate vs time

In [ ]:
# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["processed_db_dir"], extensions=[".db"], return_type="str")
conn = merge_sqlite_dbs(db_paths, source_column="db_path")
sql = """
SELECT *
FROM mmf_dataset_metadata
"""
tables_df = pd.read_sql_query(sql, conn)
conn.close()
# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)
tables_df.head()

In [ ]:
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

# Minimal inputs from your dataframe
timestamps = tables_df["image_id"].astype(np.int64).tolist()
coupling_ratio = tables_df["coupling_ratio"].astype(float).tolist()
classes = tables_df["image_device"].tolist()

def plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime("%Y-%m-%d %H:%M") for ts in timestamps]
    x_indices = list(range(len(coupling_ratio)))

    plt.figure(figsize=(10, 5))

    # 3rd-degree polynomial trend
    z = np.polyfit(x_indices, coupling_ratio, 3)
    p = np.poly1d(z)
    # plt.plot(x_indices, p(x_indices), color="red", dashes=(5, 5), linewidth=1, label="Overall Trend", zorder=3)

    custom_colors = ["C1", "C2", "C0"]
    unique_classes = sorted(set(classes))

    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [coupling_ratio[j] for j, c in enumerate(classes) if c == cls]
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=custom_colors[i % len(custom_colors)], zorder=2)

    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    plt.xticks(tick_positions, [time_labels[i] for i in tick_positions], rotation=15, ha="right")

    plt.xlabel("Time (Sequential)")
    plt.ylabel("Coupling Ratio")
    plt.title("Coupling Ratio vs Time")
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=1)

# Image file size change vs time

In [ ]:
sql = """
SELECT
    d.*,
    c.experiment_description,
    c.image_source,
    c.image_device,
    c.fiber_config,
    c.camera_config,
    c.other_config
FROM mmf_dataset_metadata AS d
LEFT JOIN mmf_experiment_config AS c
  ON c.id = d.config_id
 AND c.db_path = d.db_path;
"""

with sqlite3.connect(str(dirs["merged_db_path"])) as con:
    tables_df = pd.read_sql_query(sql, con)

In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime


temp = [
    (Path(db).parent.parent / img).as_posix()
    for db, img in zip(tables_df["db_path"], tables_df["image_path"])
]

sizes_kb = [os.path.getsize(path) / 1024 for path in temp]
timestamps = tables_df['image_id'].astype(int).tolist()  # Assuming image_id is a timestamp in seconds
classes = tables_df["image_device"].tolist()  # Assuming "image_device" is the class label

def plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime('%Y-%m-%d %H:%M') for ts in timestamps]
    x_indices = list(range(len(sizes_kb)))
    
    plt.figure(figsize=(10, 5))
    
    # Fit a 3rd-degree polynomial trend curve across all data points
    z = np.polyfit(x_indices, sizes_kb, 3)
    p = np.poly1d(z)
    plt.plot(x_indices, p(x_indices), color='red', dashes=(5, 5), linewidth=1, label='Overall Trend', zorder=3)
    
    # Define specific colors and sort classes for consistent assignment
    custom_colors = ['C1', 'C2', 'C0']
    unique_classes = sorted(list(set(classes)))
    
    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [sizes_kb[j] for j, c in enumerate(classes) if c == cls]
        
        # Select color, wrapping around if there are more than 3 classes
        color = custom_colors[i % len(custom_colors)]
        
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=color, zorder=2)
    
    # Sample the X-axis ticks
    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    selected_labels = [time_labels[i] for i in tick_positions]
    
    # Apply manual rotation and alignment
    plt.xticks(tick_positions, selected_labels, rotation=15, ha='right')
    
    plt.xlabel('Time (Sequential)')
    plt.ylabel('File Size (KB)')
    plt.title('File Size vs Time')
    
    plt.legend() 
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    
    plt.show()

# Example call:
plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=1)

# Check Abnormal samples

## Coupling ratio too big or too small samples plot
select some sample where CR smaller than 30 or bigger than 300 for inspection

## Staturation
Select and plot some saturated samples for inspection

# Other interesting samples

# Check if dataset distribution is correct/as expected 

In [ ]:
from xflow.utils.visualization import plot_image, stack_mip

data_type = "chromox_random_scan"  # chromox_random_scan  chromox_line_scan  yag_random_scan  yag_line_scan  
if data_type.startswith("chromox"):
    provider_path = config["paths"]["processed_chromox"]
    provider_db_path = provider_path + "/db/dataset_meta.db"
elif data_type.startswith("yag"):
    provider_path = config["paths"]["processed_yag"]
    provider_db_path = provider_path + "/db/dataset_meta.db"

provider = SqlProvider(
    sources={"connection": provider_db_path, "sql": config["sql"][data_type]},   
    output_config={'list': "image_path"}
)#.subsample(n_samples=10000, seed=config["seed"])

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": provider_path
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

data_pipeline = PyTorchPipeline(
    provider, 
    transforms[:-1],
).to_numpy() 


mip = stack_mip(data_pipeline[1])
plot_image(mip, title="MIP of {}".format(data_type), cmap="inferno", scale="linear") # inferno, viridis, plasma, magma, gray